# Notebook 3: Transformation and Load 


The process conducted in this notebook involves the application of an Exploratory Data Analysis on the ‘candidates’ data migrated to the MySQL database.Basically, Exploratory Data Analysis (EDA) is the process of visually and statistically summarizing, exploring, and understanding the main characteristics, patterns, and relationships within a dataset.

### Importing libraries and modules


`sys.path.append()` allows importing modules from directories that are not in the default Python search path, which is necessary in this case to reuse a module created in the “source/connection/db_utils.py” directory of the project dedicated to the database utilities, like establishing and closing the connection to the database.

In [2]:
import os
import sys
import numpy as np 
import pandas as pd
import re
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from fuzzywuzzy import process
import pycountry
sys.path.append(os.path.abspath('../src'))
from connection.db_utils import get_connection, close_connection

### Establishing the database connection

A SQLite engine object is created using the module `get_connection` located in the project´s `/src/connection/db_utils.py` script. Then, a connection object is created by connecting the engine.

In [3]:
engine = get_connection()

Engine created succesfully


### Reading the candidates data from the PosgreSQL database

The pandas library's `read_sql_table` function is employed to retrieve data directly from the PostgresSQL database table named "candidates_raw". This function requires an active database connection, represented by the connection variable, to execute the necessary SQL query. All the data from the "candidates_raw" table is then loaded into a pandas DataFrame, which is assigned to the variable ``df`. This effectively transfers the table's contents into a data structure suitable for analysis and manipulation within Python.

Following the data retrieval, the `df.head(3)` method is invoked. This command displays the first three rows of the DataFrame, providing a concise preview of the data. This allows for immediate verification that the data was successfully extracted from the SQL table and offers a quick look at the structure and initial records of the dataset, facilitating a basic understanding of the data's format and content.

In [4]:
if engine:
    df = pd.read_sql_table("candidates_raw", engine)

    if 'Application Date' in df.columns:
        df['Application Date'] = pd.to_datetime(df['Application Date'], dayfirst=True)

df.head(3)

,First Name,Last Name,Email,Application Date,Country,YOE,Seniority,Technology,Code Challenge Score,Technical Interview Score
0,Bernadette,Langworth,leonard91@yahoo.com,2021-02-26,Norway,2,Intern,Data Engineer,3,3
1,Camryn,Reynolds,zelda56@hotmail.com,2021-09-09,Panama,10,Intern,Data Engineer,2,10
2,Larue,Spinka,okey_schultz41@gmail.com,2020-04-14,Belarus,4,Mid-Level,Client Success,10,9


In [5]:
close_connection(engine)

Engine connection closed.


### Transformations

#### Column names


Database table names, column names, index names, etc. should follow a naming convention that ensures high readability and uses the English language (in general). In this case the columns follow the guidelines, but can be formated to be in lowercase letters, and their spaces replaced by underscores.

The first line of code, focuses on standardizing column names by replacing spaces with underscores. This is achieved by accessing the column names through `df.columns`, using the `.str` accessor to apply string operations element-wise, and then utilizing the `.replace(' ', '_')` method to perform the substitution. The modified column names are then reassigned back to `df.columns`.

Following this, the second line further cleans the column names by converting all characters to lowercase. The `df.rename()` method is employed with the `columns=str.lower` argument, which applies the `str.lower` function to each column name. The result is a new DataFrame with all column names in lowercase


In [6]:
df.columns = df.columns.str.replace(' ', '_')
df = df.rename(columns=str.lower)
df.rename(columns={"yoe": "years_of_experience"}, inplace=True)

df.head()

,first_name,last_name,email,application_date,country,years_of_experience,seniority,technology,code_challenge_score,technical_interview_score
0,Bernadette,Langworth,leonard91@yahoo.com,2021-02-26,Norway,2,Intern,Data Engineer,3,3
1,Camryn,Reynolds,zelda56@hotmail.com,2021-09-09,Panama,10,Intern,Data Engineer,2,10
2,Larue,Spinka,okey_schultz41@gmail.com,2020-04-14,Belarus,4,Mid-Level,Client Success,10,9
3,Arch,Spinka,elvera_kulas@yahoo.com,2020-10-01,Eritrea,25,Trainee,QA Manual,7,1
4,Larue,Altenwerth,minnie.gislason@gmail.com,2020-05-20,Myanmar,13,Mid-Level,Social Media Community Management,9,7


Now all columns are in *Snake case* , a way of writing phrases without spaces, where spaces are replaced with underscores, and the words are typically all lower case. 

#### Obtaining the hired candidates

A candidate is considered HIRED when she/he has both scores greater than or equal to 7.

In [7]:
def mark_hired_candidates(df):

    df['hired'] = (df['code_challenge_score'] >= 7) & (df['technical_interview_score'] >= 7)
    return df

df = mark_hired_candidates(df)

Now we count the total number of canditates hired and non hired.

In [8]:
num_hired = df['hired'].sum()
print(f"Total number of hired: {num_hired}")

num_not_hired = (df['hired'] == False).sum()
print(f"Total number of non-hired: {num_not_hired}")

Total number of hired: 6698
Total number of non-hired: 43302


#### Verifying input values in "seniority" and technology "columns"

A simple `for` loop to print the unique values in the seniority and technology columns of the dataset. Printing the unique values in dataset columns like "seniority" and "technology" can help verify there are no invalid or incorrect inputs. By inspecting the unique values, one can quickly identify any unexpected entries (e.g., typos, inconsistent formatting, or irrelevant values) that might cause issues in data processing or analysis.


In [9]:
def print_unique_values(df):
    columns_to_check = ['seniority', 'technology']
    
    for column in columns_to_check:
        print(f"Unique values in column '{column}':")
        print(df[column].unique())
        print()  

print_unique_values(df)

Unique values in column 'seniority':
['Intern' 'Mid-Level' 'Trainee' 'Junior' 'Lead' 'Architect' 'Senior']

Unique values in column 'technology':
['Data Engineer' 'Client Success' 'QA Manual'
 'Social Media Community Management' 'Adobe Experience Manager' 'Sales'
 'Mulesoft' 'DevOps' 'Development - CMS Backend' 'Salesforce'
 'System Administration' 'Security' 'Game Development'
 'Development - CMS Frontend' 'Security Compliance'
 'Development - Backend' 'Design'
 'Business Analytics / Project Management' 'Development - Frontend'
 'Development - FullStack' 'Business Intelligence'
 'Database Administration' 'QA Automation' 'Technical Writing']



In this cases there are not evident inlavid inputs:

- **Seniority:** All the values seem valid and consistent, representing typical seniority levels (e.g., Intern, Mid-Level, Lead, etc.). There are no obvious invalid inputs here.

- **Technology:** The entries also look reasonable, covering a wide range of roles and areas (e.g., Data Engineer, DevOps, Salesforce, etc.). There don’t appear to be typos or strange inputs.

#### Country validation

Validate and clean the 'country' column using fuzzy matching and `pycountry`.

In [10]:
def validate_and_clean_countries(df):
    # Generate the reference country list once
    reference_countries = [country.name for country in pycountry.countries]

    # Create a mapping of input countries to their corrected matches
    corrected_countries = {}
    corrections_count = 0  # Counter for the number of corrections
    corrected_country_list = set()  # To store unique corrected country names
    flagged_rows_count = 0  # Counter for unknown countries

    def correct_country(country):
        nonlocal corrections_count, flagged_rows_count  # Allow modification of the counters

        if pd.isna(country):
            flagged_rows_count += 1  # Increment the flag count for NaN entries
            return country  # Keep the original value (even if NaN)

        if country in corrected_countries:  # Use cached result if available
            return corrected_countries[country]

        match, score = process.extractOne(country, reference_countries)
        if score >= 80:  # If a valid match is found
            corrected_countries[country] = match
            if country != match:  # Count as a correction if the name changes
                corrections_count += 1
                corrected_country_list.add(match)  # Add to the list of corrected countries
            return match
        else:
            flagged_rows_count += 1  # Increment the flag count for unknown entries
            corrected_countries[country] = country  # Retain the original value
            return country

    # Correct the 'country' column directly
    df['country'] = df['country'].map(correct_country)

    # Generate a new column to flag rows with unknown countries
    df['unknown_country_flag'] = df['country'].apply(
        lambda x: pd.isna(x) or x not in reference_countries
    )

    # Print the metrics
    print(f"Total number of country names corrected: {corrections_count}")
    print(f"Total unique corrected countries: {len(corrected_country_list)}")
    print(f"List of corrected countries: {corrected_country_list}")
    print(f"Total number of rows flagged as unknown: {flagged_rows_count}")

    return df


In [11]:
df = validate_and_clean_countries(df)

Total number of country names corrected: 30
Total unique corrected countries: 29
List of corrected countries: {'Saint Barthélemy', 'Antarctica', 'Réunion', 'Taiwan, Province of China', "Côte d'Ivoire", 'Iran, Islamic Republic of', 'British Indian Ocean Territory', 'Venezuela, Bolivarian Republic of', 'Tanzania, United Republic of', 'Korea, Republic of', 'Saint Helena, Ascension and Tristan da Cunha', 'Slovakia', 'Viet Nam', "Korea, Democratic People's Republic of", 'Virgin Islands, U.S.', 'United States', 'Pitcairn', 'Virgin Islands, British', 'Netherlands', 'Micronesia, Federated States of', 'Cabo Verde', 'Moldova, Republic of', 'Svalbard and Jan Mayen', 'Bolivia, Plurinational State of', 'Saint Martin (French part)', 'North Macedonia', 'Bouvet Island', 'Libya', 'Central African Republic'}
Total number of rows flagged as unknown: 3


This output means that the dataset had 30 instances of country names that were corrected. Out of these corrections, 29 were unique country names, which indicates that one country name was corrected multiple times (likely because it appeared in the dataset more than once and was corrected in each instance).

In [12]:
unique_countries = df['country'].unique()
print(len(unique_countries))

241


After validation, there are 241 unique countries, and initially there were 244. Some entries might have been regions, territories, or other geopolitical entities that pycountry recognizes, but one might not consider as independent countries; or the names of the countries were typed in a non standard form.

##### Inspecting registers flagged with "Unknown" country

The rows (3 in this case) flagged by the `unknown_country_flag` does not contain a valid or recognizable country name, and instead has been replaced with the placeholder.This helps identify which rows might need further attention or correction.

To proceed, we extract the unique values of countries flagged as unknown.

In [13]:
unknown_countries = df[df['unknown_country_flag'] == True]['country'].unique()

print("Unique unknown countries:", unknown_countries)

Unique unknown countries: ['Palestinian Territory' 'Turkey' 'Swaziland']


**Why this happens:**

- **Swaziland:** The country officially changed its name to Eswatini in 2018. If `pycountry`´s reference list is up-to-date, older names like "Swaziland" might not match.

- **Palestinian Territory:** This might not exist in `pycountry`´s reference list as it might only contain recognized sovereign states.

- **Turkey:** The country officially requested to use its name Türkiye in international contexts instead of "Turkey."

##### Handling registers flagged with "Unknown" country

We are going to apply corrections to the country column, reeplacing outdated country names with their current, officially recognized counterparts. This is accomplished thought the Pandas `.replace()` method and a dictionary.

In [14]:
custom_corrections = {
    'Swaziland': 'Eswatini',
    'Palestinian Territory': 'State of Palestine',
    'Turkey': 'Türkiye'
}

df['country'] = df['country'].replace(custom_corrections)

Now, the column used for the validation process can be droped.

In [15]:
df = df.drop(columns=['unknown_country_flag'])
df.head(3)

,first_name,last_name,email,application_date,country,years_of_experience,seniority,technology,code_challenge_score,technical_interview_score,hired
0,Bernadette,Langworth,leonard91@yahoo.com,2021-02-26,Norway,2,Intern,Data Engineer,3,3,False
1,Camryn,Reynolds,zelda56@hotmail.com,2021-09-09,Panama,10,Intern,Data Engineer,2,10,False
2,Larue,Spinka,okey_schultz41@gmail.com,2020-04-14,Belarus,4,Mid-Level,Client Success,10,9,True


#### E-mail duplicates

As signaled before in the "Data cardinality" section, there is a duplication problem with the e-mails, as these should be unique for each candidate. But before adressing this issue, validating email addresses with Regex (regular expresions) comes first.

In [16]:
def validate_with_regex(email):

    pattern = r"^[\w\.-]+@[a-zA-Z0-9\.-]+\.[a-zA-Z]{2,}$"
    return bool(re.match(pattern, email))


def validate_email_column(df):

    df['email_valid'] = df['email'].apply(validate_with_regex)

    return df

df = validate_email_column(df)

invalid_emails = df[df['email_valid'] == False]
print("Invalid Emails:")
print(invalid_emails['email'])

Invalid Emails:
Series([], Name: email, dtype: object)


There are not invalid emails in the dataset. But there are 332 rows with duplicated emails.

In [24]:
duplicate_emails = df[df.duplicated(subset=['email'], keep=False)]
duplicate_emails["email"].value_counts()
print(f"Found {len(duplicate_emails)} rows with duplicate emails.")


Found 332 rows with duplicate emails.


##### Are there candidates applying in diferent dates?

We have to review if there are candidates applying in different dates, hence the email duplication. This is accomplished through the `chekc_duplicate_candidates` function that checks for duplicate candidates based on the combination of first name, last name, and email. It returns a DataFrame with a 'duplicated_candidate' column and prints the number of duplicates.

In [21]:
def check_duplicate_candidates(df):
 
    df['duplicated_candidate'] = df.duplicated(subset=['first_name', 'last_name', 'email'], keep=False)
    
    num_duplicates = df['duplicated_candidate'].sum()
    print(f"Number of duplicate candidates: {num_duplicates}")
    
    duplicated_records = df[df['duplicated_candidate']]
    print(duplicated_records)

    return df

df = check_duplicate_candidates(df)


Number of duplicate candidates: 0
Empty DataFrame
Columns: [first_name, last_name, email, application_date, country, years_of_experience, seniority, technology, code_challenge_score, technical_interview_score, hired, email_valid, duplicated_candidate]
Index: []


The output indicates there are not candidates applying in diferent dates.

##### Handling duplicate emails

To solve this problem using Pandas, we can follow these steps:

1. Sort the table based on the “application_date” column in ascending order (in-place sorting).


In [22]:
df.sort_values(by='application_date', ascending=True, inplace=True)
is_ordered = df['application_date'].is_monotonic_increasing
print(is_ordered)

True


2. Drop duplicate entries, keeping only the first occurrence (in-place drop duplicates), which is flagged as hired.

In [23]:
df = df[df['hired'] == True].drop_duplicates(inplace=False)

After removing duplicate entries in the dataset, the total number of rows (or "registers") left corresponds exactly to the total number of individuals hired, which is 6,698 ("Total number of hired: 6698" obtained earlier).

In [24]:
df.shape

(6698, 13)

**Implications:**

- The duplicate entries that were removed were related to individuals who were not hired or irrelevant rows in the context of those flagged as hired.
- All entries for individuals who were hired have been retained after handling duplicates, as the total number of remaining rows matches the total hired count.
- The resulting dataset is now cleaner, containing only unique records and focusing solely on hired candidates.

Finally, the columns used for validation are dropped.

In [25]:
df = df.drop(columns=['email_valid', 'hired', 'duplicated_candidate'])
df.head(3)

,first_name,last_name,email,application_date,country,years_of_experience,seniority,technology,code_challenge_score,technical_interview_score
5080,Aleen,Koelpin,krystina_marvin@yahoo.com,2018-01-01,Cayman Islands,28,Senior,QA Automation,10,9
20687,Barry,Harber,miguel_murazik@hotmail.com,2018-01-01,Mongolia,4,Mid-Level,DevOps,9,9
25573,Roselyn,Lubowitz,aaron5@gmail.com,2018-01-01,Guyana,26,Mid-Level,Client Success,8,8


### Load



A SQLite engine object is created again using the module `get_connection` located in the project´s `/src/connection/db_utils.py` script. Then, a connection object is created by connecting the engine.

In [33]:
engine = get_connection()

Engine created succesfully


A `insert_full_df` function is defined to insert data from the `df` Dataframe into a PostgreSQL database using SQLAlchemy, handling the insertion in two ways: first, it inserts the data in batches to improve performance and stability, and second, it provides a function to insert the entire DataFrame. Both functions use a SQLAlchemy engine configured with environment variables for the database connection, and handle transactions and errors in a robust way.

Its arguments are a  DataFrame (`df`), a table name (`table_name`), an SQLAlchemy engine (`engine`), and a batch size (`batch_size`) as input. It checks if the engine is available, and if so, it attempts to insert the DataFrame into the specified table in batches within a transaction. The DataFrame is divided into smaller chunks based on the batch_size, and each chunk is appended to the database table using `to_sql`. If any error occurs during the insertion of a batch, the function catches the exception, calculates the batch number where the error occurred, and prints an error message, while the transaction context ensures a rollback. 

In [36]:
def insert_full_df(df, table_name, engine, batch_size=1000):

    if engine is None:
        print("Engine not available.")
        return

    try:
        with engine.begin() as connection:  
            for i in range(0, len(df), batch_size):
                batch_df = df.iloc[i:i + batch_size]
                batch_df.to_sql(name=table_name, con=connection, if_exists='append', index=False)

        print(f"Data inserted into '{table_name}' succesfully.")

    except Exception as e:
        batch_number = i // batch_size + 1 if 'i' in locals() else 1
        print(f"Error in batch {batch_number}: {e}")



insert_full_df(df, "hirees_clean", engine)

Data inserted into 'hirees_clean' succesfully.


To close the connection to de database the module `close_connection` located in the project´s `/src/connection/db_utils.py` script is used. Its argument is the engine defined earlier.

In [37]:
close_connection(engine)

Engine connection closed.
